<a href="https://colab.research.google.com/github/coitloz88/govcrawler/blob/master/%EA%B8%88%EC%9C%B5%EC%9C%84%EC%9B%90%ED%9A%8C_%EB%B3%B4%EB%8F%84%EC%9E%90%EB%A3%8C_%EB%8B%A4%EC%9A%B4%EB%A1%9C%EB%93%9C_%EC%9E%90%EB%8F%99%ED%99%94_%EC%8B%A4%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title 1. 공식 Chrome 및 드라이버 설정
import os

# 1. 기존의 모든 크롬 관련 파일 삭제 (충돌 방지)
!apt-get remove -y google-chrome-stable chromium-browser chromium-chromedriver > /dev/null 2>&1
!rm -rf /usr/bin/google-chrome
!rm -rf /usr/bin/chromium-browser
!rm -rf /usr/bin/chromedriver

# 2. 필수 라이브러리 설치
!pip install selenium webdriver-manager -qq

# 3. Google Chrome Stable (.deb) 직접 다운로드 및 설치
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y ./google-chrome-stable_current_amd64.deb > /dev/null 2>&1

print("✅ 공식 Google Chrome 설치 완료.")

✅ 공식 Google Chrome 설치 완료.


In [ ]:
# @title 2. 금융위 사이트 접속 및 AI에게 던져줄 HTML 저장
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options

# --- [설정 영역] ---
TARGET_URL = "https://www.fsc.go.kr/no010101?curPage=2&srchCtgry=&srchEndDt=&srchKey=sj&srchBeginDt=&srchText=%EA%B0%80%EA%B3%84%EB%8C%80%EC%B6%9C+%EB%8F%99%ED%96%A5"

def get_stable_driver():
    options = Options()
    # 최신 코랩 환경에 맞춘 옵션 세트
    options.add_argument('--headless=new') # 기존 --headless 보다 안정적인 모드
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--disable-gpu')
    options.add_argument('--window-size=1920,1080')
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

    # WebDriver Manager를 통해 드라이버 자동 매칭 및 실행
    service = ChromeService(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    return driver

def create_debug_html():
    driver = None
    try:
        print("🚀 Chrome Driver(Stable) 시작 중...")
        driver = get_stable_driver()

        print(f"🌐 금융위원회 접속 중: {TARGET_URL}")
        driver.get(TARGET_URL)

        # 로딩 대기
        time.sleep(5)

        # HTML 저장
        html_source = driver.page_source
        filename = "fsc_debug.html"

        with open(filename, "w", encoding="utf-8") as f:
            f.write(html_source)

        print(f"\n✅ [성공] '{filename}' 파일이 생성되었습니다.")
        print("   좌측 폴더(📁)에서 새로고침 후 다운로드 해주세요.")

    except Exception as e:
        print(f"\n❌ 오류 발생: {e}")
        import traceback
        traceback.print_exc()

    finally:
        if driver:
            driver.quit()

if __name__ == "__main__":
    create_debug_html()

🚀 Chrome Driver(Stable) 시작 중...
🌐 금융위원회 접속 중: https://www.fsc.go.kr/no010101?curPage=2&srchCtgry=&srchEndDt=&srchKey=sj&srchBeginDt=&srchText=%EA%B0%80%EA%B3%84%EB%8C%80%EC%B6%9C+%EB%8F%99%ED%96%A5

✅ [성공] 'fsc_debug.html' 파일이 생성되었습니다.
   좌측 폴더(📁)에서 새로고침 후 다운로드 해주세요.


In [ ]:
# @title 3. 금융위원회 보도자료 자동화 작성코드
import os
import time
import requests
from urllib.parse import quote
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service as ChromeService
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options

# --- [사용자 설정 영역] ---
# 검색어: 가계대출 동향
KEYWORD = "가계대출 동향"
# 검색 기간 (YYYY-MM-DD)
START_DATE = "2024-01-01"
END_DATE = "2025-12-31"
SAVE_DIR = "fsc_loan_stats" # 저장할 폴더명
# -----------------------

def get_stable_driver():
    options = Options()
    options.add_argument('--headless=new')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--disable-gpu')
    options.add_argument('--window-size=1920,1080')
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

    service = ChromeService(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    return driver

def clean_filename(title):
    return "".join([c for c in title if c.isalpha() or c.isdigit() or c in (' ', '.', '_', '-', '(', ')', '[', ']')]).rstrip()

def download_file_hybrid(driver, download_url, file_name, save_dir):
    try:
        # Selenium 쿠키 -> Requests 이식
        cookies = driver.get_cookies()
        session = requests.Session()
        for cookie in cookies:
            session.cookies.set(cookie['name'], cookie['value'])

        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            "Referer": driver.current_url
        }

        # 절대 경로 처리
        if not download_url.startswith("http"):
            download_url = "https://www.fsc.go.kr" + download_url

        response = session.get(download_url, headers=headers, stream=True)

        if response.status_code == 200:
            file_path = os.path.join(save_dir, file_name)
            with open(file_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            print(f"      └ [성공] {file_name}")
            return True
        else:
            print(f"      └ [실패] HTTP {response.status_code}")
            return False

    except Exception as e:
        print(f"      └ [에러] {e}")
        return False

def main():
    if not os.path.exists(SAVE_DIR):
        os.makedirs(SAVE_DIR)
        print(f"📁 저장 폴더 생성: {SAVE_DIR}")

    driver = None
    total_download = 0

    try:
        driver = get_stable_driver()

        # URL 생성 (금융위 검색 패턴)
        # srchKey=sj (제목검색), srchText(검색어), srchBeginDt/EndDt(기간)
        encoded_keyword = quote(KEYWORD)
        base_url = "https://www.fsc.go.kr/no010101"

        # 페이지 순회 (일단 1~3페이지만 예시로 설정, 필요시 범위 늘리세요)
        for page in range(1, 4):
            search_url = (
                f"{base_url}?"
                f"curPage={page}&"
                f"srchKey=sj&"
                f"srchText={encoded_keyword}&"
                f"srchBeginDt={START_DATE}&"
                f"srchEndDt={END_DATE}"
            )

            print(f"\n📄 [페이지 {page}] 검색 중...")
            driver.get(search_url)
            time.sleep(3) # 로딩 대기

            # 게시글 리스트 찾기 (.board-list-wrap > ul > li)
            items = driver.find_elements(By.CSS_SELECTOR, ".board-list-wrap > ul > li")

            if not items:
                print("   ⚠️ 검색 결과가 없습니다.")
                break

            print(f"   -> {len(items)}개의 게시글 발견. 파일 스캔 시작...")

            for item in items:
                try:
                    # 제목 가져오기
                    title_el = item.find_element(By.CSS_SELECTOR, ".subject a")
                    title = title_el.text.strip()

                    # 첨부파일 리스트 확인 (.file-list)
                    # 금융위는 리스트 화면에서 바로 다운로드 가능합니다. (상세페이지 진입 불필요)
                    file_lists = item.find_elements(By.CSS_SELECTOR, ".file-list")

                    if not file_lists:
                        continue

                    print(f"   🔍 게시글: {title[:30]}...")

                    for f_item in file_lists:
                        try:
                            # 파일명 (.name)
                            f_name_text = f_item.find_element(By.CSS_SELECTOR, ".name").text

                            # PDF 파일만 대상
                            if ".pdf" in f_name_text.lower():
                                # 다운로드 버튼 (.ico.download a)
                                down_btn = f_item.find_element(By.CSS_SELECTOR, ".ico.download a")
                                down_link = down_btn.get_attribute("href")

                                # 저장할 파일명 정리
                                save_name = clean_filename(f_name_text)
                                if not save_name.lower().endswith('.pdf'):
                                    save_name += ".pdf"

                                # 중복 확인
                                if os.path.exists(os.path.join(SAVE_DIR, save_name)):
                                    print(f"      └ [스킵] 이미 있음: {save_name}")
                                    continue

                                # 다운로드 실행
                                if download_file_hybrid(driver, down_link, save_name, SAVE_DIR):
                                    total_download += 1

                        except Exception:
                            continue # 특정 파일 처리 실패 시 다음 파일로

                except Exception as e:
                    print(f"   [항목 처리 에러] {e}")
                    continue

            # 페이지당 텀
            time.sleep(1)

    except Exception as e:
        print(f"\n❌ 치명적 오류: {e}")
        import traceback
        traceback.print_exc()

    finally:
        if driver:
            driver.quit()
        print(f"\n🎉 작업 종료! 총 {total_download}개의 PDF를 다운로드했습니다.")

if __name__ == "__main__":
    main()

📁 저장 폴더 생성: fsc_loan_stats

📄 [페이지 1] 검색 중...
   -> 10개의 게시글 발견. 파일 스캔 시작...
   🔍 게시글: 2025년 11월중 가계대출 동향(잠정) 및 ｢가계부채...
   🔍 게시글: 2025년 10월중 가계대출 동향(잠정) 및 ｢가계부채...
      └ [성공] 251113(보도자료) 2025년 10월중 가계대출 동향(잠정) 및 가계부채 점검회의 개최.pdf
   🔍 게시글: 2025년 9월중 가계대출 동향(잠정)...
      └ [성공] 251016(보도자료) 2025년 9월중 가계대출 동향(잠정).pdf
   🔍 게시글: 2025년 8월중 가계대출 동향(잠정)...
      └ [성공] 250910(보도자료) 2025년 8월중 가계대출 동향(잠정).pdf
   🔍 게시글: 2025년 7월중 가계대출 동향...
      └ [성공] (250813) (보도자료) 2025년 7월중 가계대출 동향.pdf
   🔍 게시글: [보도자료] 6월중 가계대출 동향(잠정) 및 「가계부채...
      └ [성공] 250709(보도자료) 6월중 가계대출 동향(잠정) 및 가계부채 점검회의 개최.pdf
   🔍 게시글: [보도자료] 5월중 가계대출 동향(잠정) 및 「가계부채...
      └ [성공] 250611 (보도자료) 5월중 가계대출 동향(잠정) 및 가계부채 점검회의 개최.pdf
   🔍 게시글: [보도자료] 2025년 4월중 가계대출 동향(잠정) -...
      └ [성공] 250514(보도자료) 2025년 4월중 가계대출 동향(잠정).pdf
   🔍 게시글: [보도자료] 3월 중 가계대출 동향(잠정) 및 가계부채...
      └ [성공] 250409(보도자료) 3월 중 가계대출 동향(잠정) 및 가계부채 점검회의 개최.pdf
   🔍 게시글: [보도자료] 「가계부채 점검회의」 개최 - 금융권 가계...
      └ [성공] 250317(보도자료) 가계부채 점검회의 개최.pdf

📄 [페이지 2] 